# RAG: Different Ways to Chunk a PDF

## Project Scenario
This project focuses on a RAG system for a client who wants to query information from a Trustworthy AI PDF document. Multiple chunking approaches are explored to find an effective solution.


## Project Goals
Implement multiple chunking strategies (fixed-size, semantic, recursive character)
Compare chunking approaches for the PDF document
Understand how chunk size and overlap affect retrieval quality
Evaluate chunking results visually and quantitatively
Make informed recommendations about chunking strategies for the PDF document


## Project Overview
Chunking is a critical step in building effective RAG systems. The way documents are split directly impacts retrieval quality, context preservation, and overall system performance. This project explores different chunking strategies using a Trustworthy AI PDF document.


## Project Workflow


## Step 1: Setup and Data Loading


In [ ]:
%pip install  -r requirements.txt

In [ ]:
# Import required libraries
import os
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

## Step 2: Implement Fixed-Size Chunking
Objective: Create fixed-size chunks for the PDF document.

Task:

- Use CharacterTextSplitter with fixed chunk size
- Experiment with different sizes: 500, 1000, 2000 characters
- Try different overlap values: 0, 50, 100 characters
- Compare results across PDF chunking configurations

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

DATA_DIR = Path("data")

AI_ETHICS_PDF = "ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf"
PDF_PATH = DATA_DIR / AI_ETHICS_PDF

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()
pdf_text = "\n\n".join(doc.page_content for doc in documents)

print(f"Loaded {len(documents)} pages from the PDF document.")


In [ ]:
from IPython.display import Markdown, display
from langchain_text_splitters import CharacterTextSplitter

def chunk_text(text, chunk_size=500, chunk_overlap=50):
    splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separator="\n"
    )
    return splitter.split_text(text)

def chunk_documents(documents, chunk_size=500, chunk_overlap=50):
    all_chunks = []

    for i, doc in enumerate(documents):
        text = doc.page_content
        chunks = chunk_text(text, chunk_size, chunk_overlap)

        for j, chunk in enumerate(chunks):
            all_chunks.append({
                "page_content": chunk,
                "metadata": doc.metadata,
                "page_number": i + 1,
                "chunk_number": j + 1
            })

    return all_chunks

def display_full_chunks(title, chunks, count=5):
    display(Markdown(f"### {title}"))
    for index, chunk in enumerate(chunks[:count], start=1):
        display(Markdown(f"#### Chunk {index}"))
        print(chunk)
        print("\n" + "-" * 80 + "\n")


In [ ]:
from IPython.display import Markdown, display

fixed_sizes = [500, 1000, 2000]
fixed_overlaps = [0, 50, 100]

fixed_results = {}
fixed_rows = []

for chunk_size in fixed_sizes:
    for chunk_overlap in fixed_overlaps:
        chunks = chunk_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        result_key = f"size_{chunk_size}_overlap_{chunk_overlap}"
        fixed_results[result_key] = chunks

        chunk_lengths = [len(chunk["page_content"]) for chunk in chunks]

        fixed_rows.append({
            "configuration": result_key,
            "num_chunks": len(chunks),
            "avg_chunk_length": round(sum(chunk_lengths) / len(chunk_lengths), 2),
            "min_chunk_length": min(chunk_lengths),
            "max_chunk_length": max(chunk_lengths)
        })

fixed_rows = sorted(fixed_rows, key=lambda row: row["configuration"])

fixed_table_lines = [
    "| Configuration | Num Chunks | Avg Length | Min Length | Max Length |",
    "|---|---:|---:|---:|---:|"
]

for row in fixed_rows:
    fixed_table_lines.append(
        f"| {row['configuration']} | {row['num_chunks']} | {row['avg_chunk_length']} | {row['min_chunk_length']} | {row['max_chunk_length']} |"
    )

display(Markdown("\n".join(fixed_table_lines)))

fixed_preview_chunks = [chunk["page_content"] for chunk in fixed_results["size_1000_overlap_50"]]
display_full_chunks("Fixed-Size Sample Chunks (1000 / overlap 50)", fixed_preview_chunks, count=5)

### Analysis questions:

**Does fixed-size chunking break sentences in the middle?**
Yes. Fixed-size chunking often breaks PDF text in the middle of sentences because it splits only by length and does not account for sentence boundaries.

**How does it handle paragraph boundaries?**
It handles paragraph boundaries poorly. Paragraphs can be split across chunks or merged together arbitrarily, which weakens the document structure and makes sections less coherent.

**Which chunking approach handles the PDF better?**
For the PDF, structure-aware approaches generally handle the document better than a rigid fixed-size split because headings, paragraphs, and section boundaries are easier to preserve.


## Step 3: Implement Recursive Character Chunking
Objective: Use recursive character splitting, which tries to preserve semantic boundaries.

Task:

Use
RecursiveCharacterTextSplitter
(recommended by LangChain)
Experiment with chunk sizes: 500, 1000, 2000 tokens
Try different separators priority
Compare with fixed-size results


Analysis questions:
- Does recursive chunking preserve sentence boundaries better?
- How does it handle the PDF's structure?
- Does it respect PDF section headers?


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown, display

chunk_sizes = [500, 1000, 2000]

separator_configs = {
    "default_recursive": ["\n\n", "\n", ". ", " ", ""],
    "paragraph_first": ["\n\n", "\n", " ", ""],
    "sentence_first": [". ", "\n\n", "\n", " ", ""]
}

recursive_results = {}
comparison_rows = []

for chunk_size in chunk_sizes:
    for config_name, separators in separator_configs.items():
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=200,
            length_function=len,
            separators=separators
        )

        chunks = splitter.split_documents(documents)
        result_key = f"size_{chunk_size}_{config_name}"
        recursive_results[result_key] = chunks

        chunk_lengths = [len(chunk.page_content) for chunk in chunks]

        comparison_rows.append({
            "configuration": result_key,
            "num_chunks": len(chunks),
            "avg_chunk_length": round(sum(chunk_lengths) / len(chunk_lengths), 2),
            "min_chunk_length": min(chunk_lengths),
            "max_chunk_length": max(chunk_lengths)
        })

comparison_rows = sorted(comparison_rows, key=lambda row: row["configuration"])

recursive_table_lines = [
    "| Configuration | Num Chunks | Avg Length | Min Length | Max Length |",
    "|---|---:|---:|---:|---:|"
]

for row in comparison_rows:
    recursive_table_lines.append(
        f"| {row['configuration']} | {row['num_chunks']} | {row['avg_chunk_length']} | {row['min_chunk_length']} | {row['max_chunk_length']} |"
    )

display(Markdown("\n".join(recursive_table_lines)))

recursive_preview_chunks = [chunk.page_content for chunk in recursive_results["size_1000_default_recursive"]]
display_full_chunks("Recursive Sample Chunks (1000 / default separators)", recursive_preview_chunks, count=5)

### Analysis questions:

**Does recursive chunking preserve sentence boundaries better?**
Yes. Recursive chunking generally preserves sentence boundaries better than fixed-size chunking because it tries larger natural breakpoints before falling back to smaller splits.

**How does it handle the PDF's structure?**
It handles the PDF structure better than fixed-size chunking because line breaks and paragraph separators help keep related content together.

**Does it respect PDF section headers?**
Yes, usually better than fixed-size chunking. Because the splitter prioritizes paragraph and line-break separators, headings are more likely to stay attached to the content that follows.


## Step 4: Implement Token-Based Chunking
Objective: Chunk based on token count (more accurate for LLM context windows).

Task:
- Use TokenTextSplitter or calculate tokens manually
- Chunk to specific token counts (e.g., 500, 1000 tokens)
- Compare token-based vs character-based chunking

In [ ]:
from langchain_text_splitters import TokenTextSplitter
from IPython.display import Markdown, display
import tiktoken

token_sizes = [500, 1000]
token_overlap = 50

token_results = {}
token_rows = []

encoding = tiktoken.get_encoding("cl100k_base")

for chunk_size in token_sizes:
    splitter = TokenTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=token_overlap
    )

    chunks = splitter.split_documents(documents)
    result_key = f"tokens_{chunk_size}_overlap_{token_overlap}"
    token_results[result_key] = chunks

    token_counts = [len(encoding.encode(chunk.page_content)) for chunk in chunks]

    token_rows.append({
        "configuration": result_key,
        "num_chunks": len(chunks),
        "avg_tokens": round(sum(token_counts) / len(token_counts), 2),
        "min_tokens": min(token_counts),
        "max_tokens": max(token_counts)
    })

token_rows = sorted(token_rows, key=lambda row: row["configuration"])

token_table_lines = [
    "| Configuration | Num Chunks | Avg Tokens | Min Tokens | Max Tokens |",
    "|---|---:|---:|---:|---:|"
]

for row in token_rows:
    token_table_lines.append(
        f"| {row['configuration']} | {row['num_chunks']} | {row['avg_tokens']} | {row['min_tokens']} | {row['max_tokens']} |"
    )

display(Markdown("\n".join(token_table_lines)))

token_preview_chunks = [chunk.page_content for chunk in token_results["tokens_1000_overlap_50"]]
display_full_chunks("Token-Based Sample Chunks (1000 tokens / overlap 50)", token_preview_chunks, count=5)

### Comparison: Token-Based vs Character-Based Chunking

- **Precision:** Token-based chunking is more precise for LLM workflows because chunk size is measured in tokens rather than characters.
- **Consistency:** Token-based chunking produces chunk sizes that are more consistent from the model's perspective, while character-based chunks can vary significantly in token count even when their character length is similar.
- **Practicality:** Character-based chunking is simpler to implement and inspect, while token-based chunking is mainly useful when tighter context budgeting is important.
- **Recommendation:** Token-based chunking is still useful, but it is not used as often as a default strategy now that modern context windows are larger. Recursive chunking is often a more practical default when structure preservation matters.


## Step 5: Semantic Chunking (Optional - Advanced)
Note: this should be run on Colab for maximum compatibility.

Objective: Explore semantic chunking that splits based on meaning rather than size.

Task:
- Use sentence transformers to find semantic boundaries
- Split when semantic similarity drops significantly
- Compare with size-based approaches


In [ ]:
from sentence_transformers import SentenceTransformer
import re
import numpy as np
from IPython.display import Markdown, display
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_sample = pdf_text[:5000]

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_chunk(text, threshold=0.7):
    sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", text) if sentence.strip()]
    if len(sentences) < 2:
        return [text]

    embeddings = model.encode(sentences, convert_to_numpy=True, device="cpu")
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = cosine_similarity(embeddings[i - 1], embeddings[i])
        if similarity < threshold:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def summarize_text_chunks(chunks):
    lengths = [len(chunk) for chunk in chunks]
    return {
        "num_chunks": len(chunks),
        "avg_length": round(sum(lengths) / len(lengths), 2),
        "min_length": min(lengths),
        "max_length": max(lengths),
    }

def render_markdown_table(rows, columns):
    header = "| " + " | ".join(label for _, label in columns) + " |"
    divider = "|" + "|".join(["---"] + ["---:" for _ in columns[1:]]) + "|"
    lines = [header, divider]
    for row in rows:
        values = [str(row[key]) for key, _ in columns]
        lines.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))

def fixed_split(text, chunk_size=1000, chunk_overlap=50):
    splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separator="\n",
    )
    return splitter.split_text(text)

pdf_chunks_semantic = semantic_chunk(pdf_sample, threshold=0.7)

semantic_summary_rows = []

semantic_summary_rows.append({
    "method": "Semantic",
    **summarize_text_chunks(pdf_chunks_semantic),
})

fixed_sample_chunks = fixed_split(pdf_sample, chunk_size=1000, chunk_overlap=50)
semantic_summary_rows.append({
    "method": "Fixed-Size (sample)",
    **summarize_text_chunks(fixed_sample_chunks),
})

recursive_sample_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_sample_chunks = recursive_sample_splitter.split_text(pdf_sample)
semantic_summary_rows.append({
    "method": "Recursive (sample)",
    **summarize_text_chunks(recursive_sample_chunks),
})

render_markdown_table(
    semantic_summary_rows,
    [
        ("method", "Method"),
        ("num_chunks", "Num Chunks"),
        ("avg_length", "Avg Length"),
        ("min_length", "Min Length"),
        ("max_length", "Max Length"),
    ],
)

display_full_chunks("Semantic Sample Chunks", pdf_chunks_semantic, count=5)

### Implementation Note

This version uses a short PDF sample instead of the full document because semantic chunking is more computationally expensive than the previous methods. The model is also forced to run on CPU to avoid local CUDA compatibility errors in this environment.


### Comparison with Size-Based Approaches

- **Semantic:** Produces chunks based on topical or contextual shifts instead of only size limits.
- **Fixed-Size:** More predictable in length, but more likely to cut through ideas mid-way.
- **Recursive:** Better than fixed-size at preserving structure, but still primarily guided by size constraints.
- **Practical takeaway:** Semantic chunking can preserve meaning better on a sample, but it is slower and more computationally expensive than the size-based approaches.


## Step 6: Visualize and Compare Results
Objective: Create visualizations to compare chunking strategies.

Task:
- Create a comparison table of chunk statistics
- Visualize chunk size distributions
- Identify where chunks break (sentence boundaries, paragraphs, etc.)
- Document trade-offs


In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt

comparison_sets = {
    "Fixed-Size": [chunk["page_content"] for chunk in fixed_results["size_1000_overlap_50"]],
    "Recursive": [chunk.page_content for chunk in recursive_results["size_1000_default_recursive"]],
    "Token-Based": [chunk.page_content for chunk in token_results["tokens_1000_overlap_50"]],
    "Semantic (sample)": pdf_chunks_semantic
}

summary_rows = []
for method, chunks in comparison_sets.items():
    lengths = [len(chunk) for chunk in chunks]
    summary_rows.append({
        "method": method,
        "num_chunks": len(chunks),
        "avg_length": round(sum(lengths) / len(lengths), 2),
        "min_length": min(lengths),
        "max_length": max(lengths)
    })

summary_lines = [
    "| Method | Num Chunks | Avg Length | Min Length | Max Length |",
    "|---|---:|---:|---:|---:|"
]

for row in summary_rows:
    summary_lines.append(
        f"| {row['method']} | {row['num_chunks']} | {row['avg_length']} | {row['min_length']} | {row['max_length']} |"
    )

display(Markdown("\n".join(summary_lines)))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, (method, chunks) in zip(axes, comparison_sets.items()):
    lengths = [len(chunk) for chunk in chunks]
    ax.hist(lengths, bins=12, color="#4C78A8", edgecolor="white")
    ax.set_title(method)
    ax.set_xlabel("Chunk length (characters)")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

boundary_preview = [
    "| Method | Chunk Start | Chunk End |",
    "|---|---|---|"
]

for method, chunks in comparison_sets.items():
    sample_chunk = chunks[0].replace("\n", " ")
    chunk_start = sample_chunk[:80]
    chunk_end = sample_chunk[-80:]
    boundary_preview.append(
        f"| {method} | {chunk_start}... | ...{chunk_end} |"
    )

display(Markdown("\n".join(boundary_preview)))

for method, chunks in comparison_sets.items():
    display_full_chunks(f"{method} Full Chunk Samples", chunks, count=5)

### Trade-Offs Summary

- **Fixed-Size:** Simple and predictable, but more likely to break context at arbitrary points.
- **Recursive:** Better at preserving document structure and sentence flow, but slightly more complex to configure.
- **Token-Based:** More precise for LLM context windows, but less commonly used as a default when larger context windows and structure-aware strategies are available.
- **Semantic:** More meaning-aware, but computationally heavier and less practical for quick local experimentation.


## Step 7: Analyze Chunk Quality
Objective: Evaluate chunk quality by checking boundary preservation.

Task:

Check how often chunks break in the middle of sentences
Check how often chunks break in the middle of paragraphs
Identify which strategy preserves context best


In [ ]:
from IPython.display import Markdown, display
import re

quality_sets = {
    "Fixed-Size": [chunk["page_content"] for chunk in fixed_results["size_1000_overlap_50"]],
    "Recursive": [chunk.page_content for chunk in recursive_results["size_1000_default_recursive"]],
    "Token-Based": [chunk.page_content for chunk in token_results["tokens_1000_overlap_50"]],
    "Semantic (sample)": pdf_chunks_semantic,
}

sentence_boundary_pattern = re.compile(r"[.!?\]\)\"']$")

def boundary_quality(chunks):
    if len(chunks) <= 1:
        return {
            "sentence_breaks_mid": 0,
            "paragraph_breaks_mid": 0,
            "sentence_break_rate": 0.0,
            "paragraph_break_rate": 0.0,
        }

    sentence_breaks_mid = 0
    paragraph_breaks_mid = 0

    for chunk in chunks[:-1]:
        stripped = chunk.rstrip()
        if not sentence_boundary_pattern.search(stripped):
            sentence_breaks_mid += 1
        if not stripped.endswith("\n") and not stripped.endswith("\n\n"):
            paragraph_breaks_mid += 1

    total_boundaries = len(chunks) - 1

    return {
        "sentence_breaks_mid": sentence_breaks_mid,
        "paragraph_breaks_mid": paragraph_breaks_mid,
        "sentence_break_rate": round(sentence_breaks_mid / total_boundaries, 3),
        "paragraph_break_rate": round(paragraph_breaks_mid / total_boundaries, 3),
    }

quality_rows = []
for method, chunks in quality_sets.items():
    stats = boundary_quality(chunks)
    quality_rows.append({
        "method": method,
        **stats,
    })

render_markdown_table(
    quality_rows,
    [
        ("method", "Method"),
        ("sentence_breaks_mid", "Mid-Sentence Breaks"),
        ("sentence_break_rate", "Sentence Break Rate"),
        ("paragraph_breaks_mid", "Mid-Paragraph Breaks"),
        ("paragraph_break_rate", "Paragraph Break Rate"),
    ],
)

best_method = min(
    quality_rows,
    key=lambda row: (row["sentence_break_rate"], row["paragraph_break_rate"]),
)

display(Markdown(
    "### Boundary Analysis Summary\n\n"
    f"- **Mid-sentence breaks:** Lower values indicate that a strategy is less likely to cut through a sentence boundary.\n"
    f"- **Mid-paragraph breaks:** Lower values indicate better paragraph preservation.\n"
    f"- **Best overall preservation:** **{best_method['method']}** had the lowest combined boundary-break rates in this comparison."
))

## Step 8: Make Recommendations
Objective: Document your findings and make recommendations.

Task:
- Create a summary table comparing strategies
- Write recommendations for the PDF
- Explain trade-offs and when to use each approach

### Trade-offs Summary:
| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| Fixed-Size | Simple, predictable | Breaks context | Uniform content |
| Recursive | Preserves structure | More complex | Structured docs |
| Token-Based | Accurate for LLMs | Requires tokenizer | LLM integration |
| Semantic | Meaning-based | Computationally expensive | Complex content |




In [ ]:
from IPython.display import Markdown, display

strategy_rows = [
    {
        "strategy": "Fixed-Size",
        "pros": "Simple and predictable",
        "cons": "Breaks context and ignores document structure",
        "best_for": "Quick baseline chunking",
    },
    {
        "strategy": "Recursive",
        "pros": "Preserves headers, paragraphs, and sentence flow more effectively",
        "cons": "Slightly more complex to configure",
        "best_for": "Structured PDF documents",
    },
    {
        "strategy": "Token-Based",
        "pros": "Useful when strict token budgeting is important",
        "cons": "Less commonly used as a default and less intuitive to inspect than structure-aware chunking",
        "best_for": "LLM workflows with tighter context constraints",
    },
    {
        "strategy": "Semantic",
        "pros": "Can preserve topical meaning across chunk boundaries",
        "cons": "Computationally expensive and evaluated only on a sample here",
        "best_for": "Focused experiments on complex text",
    },
]

strategy_lines = [
    "| Strategy | Pros | Cons | Best For |",
    "|---|---|---|---|",
]

for row in strategy_rows:
    strategy_lines.append(
        f"| {row['strategy']} | {row['pros']} | {row['cons']} | {row['best_for']} |"
    )

display(Markdown("\n".join(strategy_lines)))

recommendation_md = """
## Chunking Strategy Recommendations

### For PDF Documents:
**Recommended Strategy:** Recursive
**Reasoning:**
- Recursive chunking preserved document structure better than fixed-size chunking.
- It was better at keeping section headers and surrounding content together.
- A configuration around 1000 characters with overlap performed well for balancing context and chunk count.

"""

display(Markdown(recommendation_md))